# Разбор и очистка одной статьи Хабра

Посмотрим путь **CSV.bz2 → HTML → обычный текст → Document → чанки** на первой статье.
Никаких эмбеддингов и запросов к LLM здесь нет.

На `home` выберите окружение `main` и запускайте ячейки сверху вниз.
Используем Light-набор из материалов курса и читаем только первую статью потоковым загрузчиком.

In [1]:
import re
import sys
from collections import Counter
from pathlib import Path
from pprint import pprint

import pandas as pd
from bs4 import BeautifulSoup

# Работает при запуске из lab_2_langchain или из third_semestr.
lab_dir = Path.cwd() if Path("chunking.py").exists() else Path.cwd() / "lab_2_langchain"
sys.path.insert(0, str(lab_dir))
from chunking import clean_html, iter_articles, load_documents, split_documents

## 1. Как устроены данные

В текущем Light-наборе это CSV.bz2: одна строка — одна статья. Поля `post_id`, `post_hubs`, `post_tags`, `post_publish_date`, `post_title`, `post_html`, `post_author`.
Загрузчик также поддерживает прежний полный JSON со структурой ниже.

```text
{
  "programming": [
    {"id": "960974", "titleHtml": "...", "textHtml": "...", ...},
    ...
  ],
  ...
}
```

`iter_articles()` читает CSV.bz2 или JSON по одной статье.
`next()` забирает только первую запись — весь список в память не загружается.

In [2]:
articles = iter_articles()
hub, article = next(articles)
articles.close()

print("Хаб:", hub)
print("ID:", article.get("post_id") or article.get("id"))
print("Заголовок:", article.get("post_title") or article.get("titleHtml"))
print("Дата:", article.get("post_publish_date") or article.get("timePublished"))
print("Полей в статье:", len(article))

Хаб: ['Sound', 'Artificial Intelligence', 'Machine learning', 'Cloud services', 'Product Management']
ID: 908346
Заголовок: SoundCloud разрешил искусственному интеллекту использовать контент пользователей
Дата: 2025-05-11T12:36:11+00:00
Полей в статье: 7


In [3]:
# Все поля первой статьи: тип и размер значения, без вывода вложенного служебного содержимого.
fields = pd.DataFrame(
    [
        {
            "поле": key,
            "тип": type(value).__name__,
            "размер": len(value) if isinstance(value, (str, list, dict)) else None,
        }
        for key, value in article.items()
    ]
)
with pd.option_context("display.max_rows", None):
    display(fields)

,поле,тип,размер
0,post_id,str,6
1,post_hubs,str,96
2,post_tags,str,34
3,post_publish_date,str,25
4,post_title,str,80
5,post_html,str,3496
6,post_author,str,7


## 2. Какие поля берём

| Поле | Что делаем |
|---|---|
| `post_title` / `titleHtml` | Очищаем и добавляем заголовок перед текстом |
| `post_html` / `textHtml` | Берём полный текст статьи и очищаем HTML |
| `leadData.textHtml` | Берём только если полного текста нет; иначе вступление может повториться |
| `id`, `timePublished`, ключ хаба | Сохраняем как метаданные, по ID строим ссылку |
| `author`, `statistics`, `relatedData` и остальные служебные поля | В текст для RAG не добавляем |

В данном примере анонс уже присутствует в начале полного текста.

In [4]:
print("Поля leadData (только в полной выгрузке):", list(article.get("leadData") or {}))
print("\nАнонс:")
print((article.get("leadData") or {}).get("textHtml", "В Light-наборе нет отдельного анонса"))

raw_html = article.get("post_html") or article.get("textHtml")
print("\nНачало полного HTML (первые 1800 символов):")
print(raw_html[:1800])

Поля leadData (только в полной выгрузке): []

Анонс:
В Light-наборе нет отдельного анонса

Начало полного HTML (первые 1800 символов):
<div xmlns="http://www.w3.org/1999/xhtml"><figure class="full-width "><img src="https://habrastorage.org/r/w1560/getpro/habr/upload_files/95d/7cd/c89/95d7cdc898332d68af71558425420d94.jpg" width="1560" height="682" sizes="(max-width: 780px) 100vw, 50vw" srcset="https://habrastorage.org/r/w780/getpro/habr/upload_files/95d/7cd/c89/95d7cdc898332d68af71558425420d94.jpg 780w,&#10;       https://habrastorage.org/r/w1560/getpro/habr/upload_files/95d/7cd/c89/95d7cdc898332d68af71558425420d94.jpg 781w" loading="lazy" decode="async"/></figure><p>SoundCloud незаметно изменил свои условия использования, чтобы позволить компании обучать ИИ на аудиозаписях, которые пользователи загружают на платформу. На это <a href="https://x.com/ednewtonrex/status/1920867088455135723?ref_src=twsrc%5Etfw%7Ctwcamp%5Etweetembed%7Ctwterm%5E1920867088455135723%7Ctwgr%5E396fa3aedfb207b9fba

## 3. Что находится внутри HTML

Разберём HTML с BeautifulSoup и посчитаем теги.
Это помогает увидеть абзацы, подзаголовки, картинки и возможные скрипты.
HTML печатаем как текст, не исполняем его в ноутбуке.

In [5]:
soup = BeautifulSoup(raw_html, "html.parser")
Counter(tag.name for tag in soup.find_all())

Counter({'p': 7, 'a': 3, 'figure': 2, 'img': 2, 'div': 1, 'blockquote': 1})

## 4. Очистка по шагам

Сначала удаляем `script`, `style` и `svg` вместе с содержимым.
В этой конкретной статье их может не быть — это общее правило загрузчика.
Затем добавляем переносы около блочных тегов, чтобы абзацы и заголовки не склеились.

In [6]:
removed = soup.find_all(["script", "style", "svg"])
print("Удаляемых элементов:", len(removed))
for tag in removed:
    tag.decompose()

block_tags = ["p", "div", "br", "li", "h1", "h2", "h3", "h4", "pre", "tr"]
for tag in soup.find_all(block_tags):
    tag.insert_before("\n")
    tag.insert_after("\n")

text = soup.get_text()
print(repr(text[:700]))  # repr показывает переносы строк как \n

Удаляемых элементов: 0
'\n\nSoundCloud незаметно изменил свои условия использования, чтобы позволить компании обучать ИИ на аудиозаписях, которые пользователи загружают на платформу. На это обратил внимание специалист по технологической этике Эд Ньютон-Рекс. \n\nПоследняя версия условий SoundCloud включает положение, разрешающее платформе использовать загруженный контент для «информирования, обучения [или] разработки» ИИ, отметил Эд Ньютон-Рекс в соцсети Х. \n\nКак пользователь SoundCloud, Эд Ньютон-Рекс не получал никаких писем с предупреждением об изменениях. По словам специалиста, он бы не согласился предоставлять свои аудиозаписи ИИ. \n\nПрофессионально лицензированная музыка от «крупных лейблов» не будет использоватьс'


`get_text()` убирает теги и декодирует HTML-сущности.
Изображения не распознаём: у `<img>` нет текстового содержимого.
Далее заменяем неразрывные пробелы и убираем невидимый символ `\u200b`,
сжимаем лишние пробелы и оставляем не более одной пустой строки подряд.

Слова, регистр и пунктуацию здесь сохраняем. Стоп-слова не удаляем:
полный читаемый текст понадобится эмбеддеру и LLM. Таблицы и отступы кода
упрощаются — это ограничение нашей учебной очистки.

In [7]:
text = text.replace("\xa0", " ").replace("\u200b", "")
text = re.sub(r"[^\S\n]+", " ", text)
text = "\n".join(line.strip() for line in text.splitlines())
cleaned_text = re.sub(r"\n{3,}", "\n\n", text).strip()

print("Символов в HTML:", len(raw_html))
print("Символов после очистки:", len(cleaned_text))
print("Совпадает с clean_html() из chunking.py:", cleaned_text == clean_html(raw_html))
print("\nОЧИЩЕННЫЙ ТЕКСТ:\n")
print(cleaned_text)

Символов в HTML: 3496
Символов после очистки: 1795
Совпадает с clean_html() из chunking.py: True

ОЧИЩЕННЫЙ ТЕКСТ:

SoundCloud незаметно изменил свои условия использования, чтобы позволить компании обучать ИИ на аудиозаписях, которые пользователи загружают на платформу. На это обратил внимание специалист по технологической этике Эд Ньютон-Рекс.

Последняя версия условий SoundCloud включает положение, разрешающее платформе использовать загруженный контент для «информирования, обучения [или] разработки» ИИ, отметил Эд Ньютон-Рекс в соцсети Х.

Как пользователь SoundCloud, Эд Ньютон-Рекс не получал никаких писем с предупреждением об изменениях. По словам специалиста, он бы не согласился предоставлять свои аудиозаписи ИИ.

Профессионально лицензированная музыка от «крупных лейблов» не будет использоваться в процессах, связанных с искусственным интеллектом, пишет TechCrunch. Представитель SoundCloud прислал изданию письмо, в котором пояснил:

«SoundCloud никогда не использовал контент испол

## 5. Собираем Document

Основной загрузчик делает такую же очистку, добавляет заголовок и метаданные.
Для корпуса он также пропускает пустые записи и дубли по ID или хешу
очищенного текста с заголовком. Здесь загружаем одну статью, поэтому
в статистике дублей ожидаемо нет. Исходный файл не изменяется.

In [8]:
documents, stats = load_documents(limit=1)
document = documents[0]

print("Статистика загрузки:")
pprint(stats)
print("\nМетаданные (они не являются телом статьи):")
pprint(document.metadata)
print("\nНачало page_content: заголовок + очищенный текст\n")
print(document.page_content[:800])

Статистика загрузки:
{'documents': 1, 'duplicates': 0, 'empty': 0, 'lead_only': 0, 'scanned': 1}

Метаданные (они не являются телом статьи):
{'article_id': '908346',
 'date': '2025-05-11T12:36:11+00:00',
 'hub': "['Sound', 'Artificial Intelligence', 'Machine learning', 'Cloud "
        "services', 'Product Management']",
 'lead_only': False,
 'source': 'https://habr.com/ru/articles/908346/',
 'title': 'SoundCloud разрешил искусственному интеллекту использовать контент '
          'пользователей'}

Начало page_content: заголовок + очищенный текст

SoundCloud разрешил искусственному интеллекту использовать контент пользователей

SoundCloud незаметно изменил свои условия использования, чтобы позволить компании обучать ИИ на аудиозаписях, которые пользователи загружают на платформу. На это обратил внимание специалист по технологической этике Эд Ньютон-Рекс.

Последняя версия условий SoundCloud включает положение, разрешающее платформе использовать загруженный контент для «информирования, о

## 6. Рекурсивный чанкинг

`split_documents()` использует `RecursiveCharacterTextSplitter` из LangChain.
Размер — **1000 символов**, целевое перекрытие — **150 символов**.
Сплиттер сначала старается разделять по абзацам, затем по строкам и пробелам.
Фактическое перекрытие не обязано быть ровно 150 символов.
Метаданные статьи сохраняются в каждом чанке.

In [9]:
chunks = split_documents(documents, chunk_size=1000, chunk_overlap=150)
print("Получилось чанков:", len(chunks))
pd.DataFrame(
    [
        {
            "чанк": i,
            "символов": len(chunk.page_content),
            "начало в исходном тексте": chunk.metadata["start_index"],
        }
        for i, chunk in enumerate(chunks, start=1)
    ]
)

Получилось чанков: 2


,чанк,символов,начало в исходном тексте
0,1,916,0
1,2,959,918


In [10]:
for i, chunk in enumerate(chunks, start=1):
    print(f"\n--- ЧАНК {i}: {len(chunk.page_content)} символов ---\n")
    print(chunk.page_content)


--- ЧАНК 1: 916 символов ---

SoundCloud разрешил искусственному интеллекту использовать контент пользователей

SoundCloud незаметно изменил свои условия использования, чтобы позволить компании обучать ИИ на аудиозаписях, которые пользователи загружают на платформу. На это обратил внимание специалист по технологической этике Эд Ньютон-Рекс.

Последняя версия условий SoundCloud включает положение, разрешающее платформе использовать загруженный контент для «информирования, обучения [или] разработки» ИИ, отметил Эд Ньютон-Рекс в соцсети Х.

Как пользователь SoundCloud, Эд Ньютон-Рекс не получал никаких писем с предупреждением об изменениях. По словам специалиста, он бы не согласился предоставлять свои аудиозаписи ИИ.

Профессионально лицензированная музыка от «крупных лейблов» не будет использоваться в процессах, связанных с искусственным интеллектом, пишет TechCrunch. Представитель SoundCloud прислал изданию письмо, в котором пояснил:

--- ЧАНК 2: 959 символов ---

«SoundCloud никогда н

Теперь можно менять `chunk_size` и `chunk_overlap` в ячейке выше и смотреть,
где проходят границы. Эти чанки затем поступят в FAISS и BM25.
В приложении индексируем первые **500** уникальных статей, а на вопрос
возвращаем **10** чанков — это два разных ограничения.